## Py17. Transform Orders Data - Explode Arrays — objective

- Compare raw fields with typed records and parsing failures.
- Follow nested JSON and arrays as they become columns or rows.

Historical results; no code executed.
- [Original inputs: gizmobox/orders](../../../data/gizmobox/orders/) — 4 files.

## Learning notes — corrective commentary (originals unchanged)

Supplied orders JSONL includes malformed lines; inspect corrupt-record handling before interpreting row counts. Original files are linked above.

# Py17. Transform Orders Data - Explode Arrays

Read the original source and historical results; nothing was executed.

[Full original payload](../../../artifacts/notebook-payloads/dea03-etl-with-apache-spark/Py17.%20Transform%20Orders%20Data%20-%20Explode%20Arrays.json)

## Learning notes — before source operation

**json-array**: JSON extraction and array operations change structure: parsing creates typed values, while explode can create multiple rows per input. Inspect missing fields, null arrays and empty arrays.

## Transform Orders Data - Explode Arrays
1. Access elements from the JSON object
1. Deduplicate Array Elements
1. Explode Arrays
1. Write the Transformed Data to Silver Schema

In [ ]:
df_orders = spark.table('gizmobox.silver.py_orders_json')
display(df_orders)

json_value
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]
[nested value omitted]


### 1. Access elements from the JSON object

In [ ]:
from pyspark.sql import functions as F

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

In [ ]:
df_orders_normalized = (
    df_orders
    .select (
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        "json_value.items"
    )
)
display(df_orders_normalized)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,items
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,[nested value omitted]
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,[nested value omitted]
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,[nested value omitted]
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,[nested value omitted]
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,[nested value omitted]
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,[nested value omitted]
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,[nested value omitted]
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,[nested value omitted]
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,[nested value omitted]
29,Pending,Bank Transfer,129,2025-01-18 03:05:04,5953,[nested value omitted]


### 2. Deduplicate Array Elements

In [ ]:
df_orders_normalized = (
    df_orders
    .select (
        "json_value.order_id",
        "json_value.order_status",
        "json_value.payment_method",
        "json_value.total_amount",
        "json_value.transaction_timestamp",
        "json_value.customer_id",
        F.array_distinct("json_value.items").alias("items")
    )
)
display(df_orders_normalized)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,items
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,[nested value omitted]
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,[nested value omitted]
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,[nested value omitted]
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,[nested value omitted]
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,[nested value omitted]
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,[nested value omitted]
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,[nested value omitted]
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,[nested value omitted]
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,[nested value omitted]
29,Pending,Bank Transfer,129,2025-01-18 03:05:04,5953,[nested value omitted]


### 3. Explode Arrays

In [ ]:
df_orders_exploded = (
    df_orders_normalized
    .select (
        "order_id",
        "order_status",
        "payment_method",
        "total_amount",
        "transaction_timestamp",
        "customer_id",
        F.explode("items").alias("item")
    )
)

display(df_orders_exploded)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,[nested value omitted]
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,[nested value omitted]
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,[nested value omitted]
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,[nested value omitted]
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,[nested value omitted]
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,[nested value omitted]
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,[nested value omitted]
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,[nested value omitted]
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,[nested value omitted]
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,[nested value omitted]


In [ ]:
df_order_items = (
    df_orders_exploded
    .select (
        "order_id",
        "order_status",
        "payment_method",
        "total_amount",
        "transaction_timestamp",
        "customer_id",
        "item.item_id",
        "item.name",
        "item.price",
        "item.quantity",
        "item.category",
        "item.details.brand",
        "item.details.color"
    )
)
display(df_order_items)

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item_id,name,price,quantity,category,brand,color
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,8,Gaming Console,499,1,Electronics,Sony,Blue
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,4,Smartwatch,299,2,Electronics,Canon,Gray
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,7,External Hard Drive,129,3,Electronics,Dell,Blue
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,3,Wireless Headphones,199,3,Electronics,Apple,White
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,2,Laptop,999,1,Electronics,Microsoft,Black
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,5,Tablet,399,1,Electronics,LG,White
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,3,Wireless Headphones,199,3,Electronics,Bose,Blue
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,10,Drone,799,1,Electronics,Dell,Black
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,3,Wireless Headphones,199,1,Electronics,HP,Silver
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,1,Smartphone,699,1,Electronics,Samsung,Silver


### 4. Write the Transformed Data to Silver Schema

In [ ]:
df_order_items.writeTo("gizmobox.silver.py_orders").createOrReplace()

Archived empty results: 1. Full archived values are in the linked payload reference in the notebook header.

Archived Databricks source — shown for reading, not Python execution.

```sql
%sql
SELECT * FROM gizmobox.silver.py_orders;
```

In [ ]:
# Archived output only; no computation.

order_id,order_status,payment_method,total_amount,transaction_timestamp,customer_id,item_id,name,price,quantity,category,brand,color
1,Completed,Bank Transfer,499,2025-01-05 10:13:59,6973,8,Gaming Console,499,1,Electronics,Sony,Blue
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,4,Smartwatch,299,2,Electronics,Canon,Gray
2,Cancelled,PayPal,985,2025-01-19 00:05:13,3532,7,External Hard Drive,129,3,Electronics,Dell,Blue
3,Completed,Bank Transfer,597,2025-01-08 23:11:00,3532,3,Wireless Headphones,199,3,Electronics,Apple,White
4,Cancelled,Bank Transfer,999,2025-01-05 05:49:26,3532,2,Laptop,999,1,Electronics,Microsoft,Black
8,Completed,PayPal,399,2025-01-02 13:11:15,9179,5,Tablet,399,1,Electronics,LG,White
14,Cancelled,PayPal,597,2025-01-01 18:22:26,7207,3,Wireless Headphones,199,3,Electronics,Bose,Blue
18,Pending,Credit Card,799,2025-01-22 21:12:08,4468,10,Drone,799,1,Electronics,Dell,Black
22,Pending,PayPal,199,2025-01-01 15:49:19,4761,3,Wireless Headphones,199,1,Electronics,HP,Silver
26,Shipped,Bank Transfer,699,2025-01-05 19:01:25,7007,1,Smartphone,699,1,Electronics,Samsung,Silver
